# Recomendação — similaridade entre países e pares sem histórico

Lê os dois arquivos gerados pelos notebooks de coleta e produz o ranking final,
incluindo os pares (NCM, país) que nunca tiveram exportação registrada.

- `ranking.csv` — 245 pares já classificados e pontuados (`Dados_Comex.ipynb`)
- `indicadores_paises.csv` — PIB per capita PPC, população e IDH (`WorldBank.ipynb`)

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors

In [4]:
ranking = pd.read_csv("ranking.csv", encoding='utf-8-sig', dtype={'Código NCM': str})
indicadores = pd.read_csv("ColetarDados/indicadores_paises.csv", encoding='utf-8-sig')

print("ranking:", ranking.shape)
print("indicadores:", indicadores.shape)
print()
print(ranking['caso'].value_counts().to_string())

ranking: (245, 7)
indicadores: (5, 4)

caso
intermitente       130
nunca exportou      71
continuidade        28
surgimento          11
descontinuidade      5


## 1. Similaridade entre países (KNN, K = 1)

Três atributos, todos de 2023: PIB per capita PPC, população e IDH.

A população entra em **escala logarítmica**. Os cinco países vão de 231 mil habitantes
(São Tomé) a 36,7 milhões (Angola); na escala linear, depois da normalização, os três
menores ficam todos próximos de zero e deixam de ser distinguíveis entre si.

K = 1 é uma escolha consciente: com cinco países, cada um tem apenas quatro vizinhos
possíveis, e valores maiores de K tomariam metade do conjunto como vizinhança.

In [5]:
indicadores['populacao_log'] = np.log10(indicadores['populacao'])

colunas_atributos = ['pib_per_capita', 'populacao_log', 'idh']

escalador = MinMaxScaler()
atributos = escalador.fit_transform(indicadores[colunas_atributos])

tabela_atributos = pd.DataFrame(atributos,
                                columns=colunas_atributos,
                                index=indicadores['pais'])
print(tabela_atributos.round(3))

                     pib_per_capita  populacao_log    idh
pais                                                     
Angola                        0.918          1.000  0.703
Cabo Verde                    1.000          0.161  1.000
Guiné-Bissau                  0.149          0.440  0.120
Moçambique                    0.000          0.983  0.000
São Tomé e Príncipe           0.522          0.000  0.823


In [6]:
# n_neighbors=2 porque o primeiro vizinho que o sklearn retorna é o próprio país
modelo = NearestNeighbors(n_neighbors=2, metric='euclidean')
modelo.fit(atributos)

distancias, indices = modelo.kneighbors(atributos)

vizinho_de = {}
distancia_do_vizinho = {}

for i in range(len(indicadores)):
    pais = indicadores.loc[i, 'pais']
    indice_vizinho = indices[i][1]
    vizinho_de[pais] = indicadores.loc[indice_vizinho, 'pais']
    distancia_do_vizinho[pais] = round(distancias[i][1], 3)

for pais in vizinho_de:
    print(pais.ljust(22), "->", vizinho_de[pais].ljust(22), distancia_do_vizinho[pais])

Angola                 -> Cabo Verde             0.894
Cabo Verde             -> São Tomé e Príncipe    0.534
Guiné-Bissau           -> Moçambique             0.575
Moçambique             -> Guiné-Bissau           0.575
São Tomé e Príncipe    -> Cabo Verde             0.534


## 2. Herança para os pares sem histórico

Para cada par (NCM, país) classificado como "nunca exportou", olha-se o país mais
parecido. Se esse vizinho tem histórico daquele produto, o par herda uma fração da
nota dele.

`TETO_HERDADO = 0.9` mantém toda nota herdada **abaixo de 1,0**, que é o piso do caso
intermitente. Ou seja: evidência indireta nunca ultrapassa evidência direta. Os 71 pares
passam a ser ordenados entre si, mas nenhum deles fica à frente de um país que
efetivamente exportou aquele produto.

Quando o vizinho também não tem histórico do produto, o par permanece em zero — é o
limite de K = 1.

In [7]:
SCORE_MAXIMO = 3.8     # peso da continuidade (3.0) + margem (0.8)
TETO_HERDADO = 0.9     # logo abaixo do piso do caso intermitente

# consulta rápida: (ncm, país) -> score e caso
score_de = {}
caso_de = {}
for i in range(len(ranking)):
    chave = (ranking.loc[i, 'Código NCM'], ranking.loc[i, 'Países'])
    score_de[chave] = ranking.loc[i, 'score']
    caso_de[chave] = ranking.loc[i, 'caso']


scores_finais = []
origens = []

for i in range(len(ranking)):
    ncm = ranking.loc[i, 'Código NCM']
    pais = ranking.loc[i, 'Países']
    caso = ranking.loc[i, 'caso']

    if caso != "nunca exportou":
        scores_finais.append(ranking.loc[i, 'score'])
        origens.append("")
        continue

    vizinho = vizinho_de[pais]
    chave_vizinho = (ncm, vizinho)

    score_vizinho = 0.0
    if chave_vizinho in score_de:
        score_vizinho = score_de[chave_vizinho]

    if score_vizinho > 0:
        nota = score_vizinho / SCORE_MAXIMO * TETO_HERDADO
        scores_finais.append(round(nota, 3))
        origens.append(vizinho + " (" + caso_de[chave_vizinho] + ")")
    else:
        scores_finais.append(0.0)
        origens.append("")

ranking['score_final'] = scores_finais
ranking['herdado_de'] = origens

sem_historico = ranking[ranking['caso'] == "nunca exportou"]
print("Pares sem histórico:", len(sem_historico))
print("  receberam nota do vizinho:", (sem_historico['score_final'] > 0).sum())
print("  continuam em zero:", (sem_historico['score_final'] == 0).sum())

Pares sem histórico: 71
  receberam nota do vizinho: 39
  continuam em zero: 32


## 3. Ranking final

In [8]:
ranking = ranking.sort_values(['Código NCM', 'score_final'], ascending=[True, False])
ranking['posicao_final'] = ranking.groupby('Código NCM').cumcount() + 1

ranking['score_final_0_10'] = (ranking['score_final'] / SCORE_MAXIMO * 10).round(1)

print(ranking.shape)
ranking.head(10)

(245, 11)


,Código NCM,Descrição NCM,Países,caso,score,posicao,score_0_10,score_final,herdado_de,posicao_final,score_final_0_10
0,07129090,Outros produtos hortícolas; misturas de produt...,Guiné-Bissau,surgimento,2.500,1,6.6,2.500,,1,6.6
1,07129090,Outros produtos hortícolas; misturas de produt...,Angola,intermitente,1.527,2,4.0,1.527,,2,4.0
2,07129090,Outros produtos hortícolas; misturas de produt...,Moçambique,intermitente,1.000,3,2.6,1.000,,3,2.6
3,07129090,Outros produtos hortícolas; misturas de produt...,Cabo Verde,nunca exportou,0.000,4,0.0,0.000,,4,0.0
4,07129090,Outros produtos hortícolas; misturas de produt...,São Tomé e Príncipe,nunca exportou,0.000,5,0.0,0.000,,5,0.0
5,08011100,"Cocos, frescos ou secos, dessecados",Angola,continuidade,3.622,1,9.5,3.622,,1,9.5
6,08011100,"Cocos, frescos ou secos, dessecados",Cabo Verde,intermitente,1.645,2,4.3,1.645,,2,4.3
7,08011100,"Cocos, frescos ou secos, dessecados",Moçambique,intermitente,1.248,3,3.3,1.248,,3,3.3
8,08011100,"Cocos, frescos ou secos, dessecados",Guiné-Bissau,intermitente,1.000,4,2.6,1.000,,4,2.6
9,08011100,"Cocos, frescos ou secos, dessecados",São Tomé e Príncipe,intermitente,1.000,5,2.6,1.000,,5,2.6


## 4. Consultas

In [9]:
# o que mudou no topo dos rankings
antes = ranking[ranking['posicao'] == 1]['Países'].value_counts()
depois = ranking[ranking['posicao_final'] == 1]['Países'].value_counts()

comparacao = pd.DataFrame({'antes': antes, 'depois': depois}).fillna(0).astype(int)
comparacao

,antes,depois
Países,,
Angola,35,35
Cabo Verde,9,9
Guiné-Bissau,3,3
Moçambique,2,2


In [10]:
# os pares que saíram do zero, do maior para o menor
recuperados = ranking[(ranking['caso'] == "nunca exportou") & (ranking['score_final'] > 0)]
recuperados = recuperados.sort_values('score_final', ascending=False)

print("Pares recuperados:", len(recuperados))
recuperados[['Código NCM', 'Descrição NCM', 'Países', 'score_final', 'herdado_de']].head(20)

Pares recuperados: 39


,Código NCM,Descrição NCM,Países,score_final,herdado_de
184,20098990,"Sucos (sumo) de outras frutas, não fermentado,...",São Tomé e Príncipe,0.893,Cabo Verde (continuidade)
209,21041011,"Preparações para caldos e sopas, em embalagens...",São Tomé e Príncipe,0.816,Cabo Verde (continuidade)
229,21069029,"Pós para preparações de cremes, sorvetes, flan...",São Tomé e Príncipe,0.781,Cabo Verde (continuidade)
124,17049020,"Caramelos, confeitos, dropes, pastilhas, e pro...",Guiné-Bissau,0.774,Moçambique (continuidade)
109,12119090,"Outras plantas e partes, para perfumaria, medi...",São Tomé e Príncipe,0.744,Cabo Verde (surgimento)
144,19012090,Outras misturas e pastas para a preparação de ...,São Tomé e Príncipe,0.592,Cabo Verde (surgimento)
19,09041200,"Pimenta (do gênero Piper), triturada ou em pó",Moçambique,0.592,Guiné-Bissau (surgimento)
164,19030000,Tapioca e seus sucedâneos preparados a partir ...,São Tomé e Príncipe,0.400,Cabo Verde (intermitente)
84,11029000,Farinha de outros cereais,São Tomé e Príncipe,0.390,Cabo Verde (intermitente)
134,18061000,"Cacau em pó, com adição de açúcar ou outros ed...",Guiné-Bissau,0.381,Moçambique (intermitente)


In [11]:
# onde cada país tem mais oportunidade vinda da similaridade
recuperados['Países'].value_counts()

Países
São Tomé e Príncipe    18
Guiné-Bissau           13
Moçambique              4
Cabo Verde              4
Name: count, dtype: int64

In [12]:
# ranking completo de um produto
ranking[ranking['Código NCM'] == '17019900'][
    ['Países', 'caso', 'score_final', 'score_final_0_10', 'posicao_final', 'herdado_de']
]

,Países,caso,score_final,score_final_0_10,posicao_final,herdado_de
115,Cabo Verde,continuidade,3.800,10.0,1,
116,São Tomé e Príncipe,surgimento,3.140,8.3,2,
117,Guiné-Bissau,continuidade,3.089,8.1,3,
118,Angola,continuidade,3.000,7.9,4,
119,Moçambique,intermitente,1.688,4.4,5,


In [13]:
# os 32 pares que continuam sem recomendação: o vizinho também não tem histórico
orfaos = ranking[(ranking['caso'] == "nunca exportou") & (ranking['score_final'] == 0)]
print("Pares sem recomendação:", len(orfaos))
orfaos['Países'].value_counts()

Pares sem recomendação: 32


Países
Cabo Verde             8
São Tomé e Príncipe    8
Guiné-Bissau           8
Moçambique             8
Name: count, dtype: int64

## 5. Salvar

In [14]:
ranking.to_csv("ranking_final.csv", index=False, encoding='utf-8-sig')
print("ranking_final.csv:", ranking.shape)

ranking_final.csv: (245, 11)
